In [3]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Output, Tab, Accordion
from IPython.display import display

# ==========================================
# 1. DESIGN SYSTEM & CONFIGURATION
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif"
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange
COLOR_NAVY = "#0B192C"          # Deep Navy Header
COLOR_SUCCESS = "#2ECC71"       # Green Baseline
COLOR_DANGER = "#E74C3C"        # Alert Red
COLOR_WARNING = "#F39C12"       # Warning Orange
COLOR_PURPLE = "#8E44AD"        # System Cancel / Metric Purple
COLOR_GREY = "#95A5A6"          # Neutral Grey
COLOR_BLUE_BAR = "#5DADE2"       # Dispatch Pickup Bar Color

SERVICE_CONFIG = {
    'SIEU_TOC': {
        'name': 'Siêu Tốc', 
        'desc': 'Gán tức thời (Target TTP < 10p)', 
        'p50_target': 5.0, 'p90_target': 9.0, 'tipping_point': 12.0, 
        'cancel_alert_threshold': 5.0, 'grid_pos': (1, 1),
        'group': 'ON_DEMAND'
    },
    'NHANH': {
        'name': 'Nhanh', 
        'desc': 'Gán nhanh (Target TTP < 15p)', 
        'p50_target': 7.0, 'p90_target': 13.0, 'tipping_point': 18.0, 
        'cancel_alert_threshold': 6.5, 'grid_pos': (1, 2),
        'group': 'ON_DEMAND'
    },
    '4H': {
        'name': '4H', 
        'desc': 'Gom chuyến (Target TTP < 30p)', 
        'p50_target': 15.0, 'p90_target': 28.0, 'tipping_point': 38.0, 
        'cancel_alert_threshold': 8.5, 'grid_pos': (2, 1),
        'group': 'BATCHING'
    },
    'DONG_GIA': {
        'name': 'Đồng Giá', 
        'desc': 'Gom chuyến linh hoạt (Target TTP < 40p)', 
        'p50_target': 20.0, 'p90_target': 35.0, 'tipping_point': 50.0, 
        'cancel_alert_threshold': 7.5, 'grid_pos': (2, 2),
        'group': 'BATCHING'
    }
}

HCM_DISTRICTS = [
    'Quận 1', 'Quận 3', 'Quận 4', 'Quận 5', 'Quận 6', 'Quận 7', 'Quận 8', 'Quận 10', 
    'Quận 11', 'Quận 12', 'Tân Bình', 'Tân Phú', 'Bình Thạnh', 'Phú Nhuận', 'Gò Vấp', 
    'Bình Tân', 'Thủ Đức', 'Bình Chánh', 'Hóc Môn', 'Nhà Bè', 'Củ Chi', 'Cần Giờ'
]

HOUR_SLOTS = ['ALL', '00:00 - 06:00', '06:00 - 12:00', '12:00 - 18:00', '18:00 - 24:00']

# B2B Merchant Master Dataset (Matching Screenshot 3)
B2B_MERCHANTS_MASTER = [
    {"name": "Kho Shopee Express - KCN Vĩnh Lộc", "district": "Bình Tân", "time_batching": 49.2, "time_ondemand": 16.4},
    {"name": "Kho Boxme - KCN Cát Lái", "district": "Thủ Đức", "time_batching": 46.5, "time_ondemand": 15.5},
    {"name": "Tổng kho Lazada - KCN Tân Bình", "district": "Tân Bình", "time_batching": 42.8, "time_ondemand": 14.3},
    {"name": "Kho ViettelPost - Sống Thần", "district": "Thủ Đức", "time_batching": 39.4, "time_ondemand": 13.1},
    {"name": "Kho Tiki Logistics - KCN Tân Thuận", "district": "Quận 7", "time_batching": 37.6, "time_ondemand": 12.5},
    {"name": "Kho Ninjavan - Bình Tân", "district": "Bình Tân", "time_batching": 32.1, "time_ondemand": 10.7},
    {"name": "Kho GHN - Tân Phú", "district": "Tân Phú", "time_batching": 28.5, "time_ondemand": 9.5},
    {"name": "Kho Sendo - Q12", "district": "Quận 12", "time_batching": 24.8, "time_ondemand": 8.3},
    {"name": "Kho Pharmacity Hub - Bình Dương", "district": "Bình Dương", "time_batching": 21.3, "time_ondemand": 7.1},
    {"name": "Kho BEST Express - Củ Chi", "district": "Củ Chi", "time_batching": 19.5, "time_ondemand": 6.5}
]

CSS_STYLING = HTML("""
<style>
    .card-container { display: flex; gap: 12px; margin-bottom: 12px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
    .kpi-card { flex: 1; background: #FFFFFF; border: 1px solid #E2E8F0; border-radius: 8px; padding: 12px 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
    .kpi-card-green { border-left: 5px solid #2ECC71; }
    .kpi-card-red { border-left: 5px solid #E74C3C; }
    .kpi-card-orange { border-left: 5px solid #F39C12; }
    .kpi-card-purple { border-left: 5px solid #8E44AD; }
    .kpi-card-black { border-left: 5px solid #1A202C; }
    .kpi-title { font-size: 11px; font-weight: 700; color: #718096; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 4px; }
    .kpi-value { font-size: 22px; font-weight: 800; color: #1A202C; margin-bottom: 2px; }
    .kpi-value-red { font-size: 22px; font-weight: 800; color: #E74C3C; margin-bottom: 2px; }
    .kpi-value-green { font-size: 22px; font-weight: 800; color: #2ECC71; margin-bottom: 2px; }
    .kpi-sub { font-size: 12px; color: #4A5568; line-height: 1.4; }
    .global-filter-banner { background: #FFF9E6; border: 1px solid #FFE58F; border-radius: 6px; padding: 6px 12px; font-size: 12px; color: #8C6B00; display: inline-flex; align-items: center; gap: 6px; }
    .action-btn-orange { background-color: #FF6B00 !important; color: white !important; font-weight: bold !important; border-radius: 4px !important; }
</style>
""")

# ==========================================
# 2. DATA ENGINE FOR YEAR 2025 (HCM FOCUS)
# ==========================================
np.random.seed(2025)

def generate_base_data_2025():
    dates = pd.date_range('2025-01-01', '2025-12-31', freq='D')
    records = []
    
    dist_vol_weights = {
        'Quận 1': 14.7, 'Tân Bình': 14.2, 'Quận 3': 13.5, 'Quận 10': 12.9, 'Gò Vấp': 12.7, 
        'Bình Thạnh': 12.4, 'Quận 7': 12.1, 'Tân Phú': 10.5, 'Phú Nhuận': 9.6, 'Quận 5': 9.2, 
        'Quận 11': 8.8, 'Quận 8': 7.8, 'Quận 4': 7.6, 'Quận 6': 7.0, 'Quận 12': 6.6, 
        'Bình Tân': 6.1, 'Thủ Đức': 4.8, 'Bình Chánh': 4.7, 'Hóc Môn': 4.5, 'Nhà Bè': 3.3, 
        'Củ Chi': 3.0, 'Cần Giờ': 1.9
    }

    hour_slots = ['00:00 - 06:00', '06:00 - 12:00', '12:00 - 18:00', '18:00 - 24:00']
    
    for dt in dates:
        day_str = dt.strftime('%Y-%m-%d')
        month_mult = 1.35 if dt.month in [1, 2] else (1.2 if dt.month in [9, 10] else 1.0)
        
        for dist, weight in dist_vol_weights.items():
            for hr_idx, hour_slot in enumerate(hour_slots):
                slot_mult = 0.12 if hr_idx == 0 else (0.38 if hr_idx == 1 else (0.32 if hr_idx == 2 else 0.18))
                
                for s_id, cfg in SERVICE_CONFIG.items():
                    base_vol = int(weight * 35 * slot_mult * month_mult)
                    if s_id == 'SIEU_TOC': base_vol = int(base_vol * 1.5)
                    elif s_id == 'DONG_GIA': base_vol = int(base_vol * 0.5)
                    base_vol = max(3, base_vol)

                    p50_val = round(np.random.normal(cfg['p50_target'], 0.5), 1)
                    p90_val = round(cfg['p90_target'] + np.random.normal(0, 1.5), 1)
                    is_lta_alert = (p90_val > cfg['tipping_point'])

                    if dist in ['Quận 1', 'Tân Bình', 'Quận 10'] and s_id in ['SIEU_TOC', 'NHANH']:
                        breach_pct = round(np.random.uniform(28.0, 42.0), 1)
                    elif dist in ['Quận 7', 'Bình Tân', 'Củ Chi'] and s_id in ['4H', 'DONG_GIA']:
                        breach_pct = round(np.random.uniform(35.0, 48.0), 1)
                    else:
                        breach_pct = round(np.random.uniform(3.0, 14.0), 1)
                        
                    breach_vol = int(base_vol * (breach_pct / 100))

                    u_cancel = round(np.random.uniform(2.0, 4.5), 2)
                    d_cancel = round(np.random.uniform(0.8, 2.0), 2)
                    s_cancel = round(np.random.uniform(0.3, 1.2), 2)
                    tot_cancel = round(u_cancel + d_cancel + s_cancel, 2)
                    is_cancel_alert = (tot_cancel > cfg['cancel_alert_threshold'])

                    pickup_dist = round(np.random.uniform(0.8, 2.8), 2)
                    reject_rate = round(np.random.uniform(8.0, 26.0), 1)

                    records.append({
                        'date': dt,
                        'day_str': day_str,
                        'month': dt.month,
                        'hour_slot': hour_slot,
                        'hour_peak': 8 if hr_idx == 1 else (17 if hr_idx == 2 else 12),
                        'district': dist,
                        'service_id': s_id,
                        'service_group': cfg['group'],
                        'volume': base_vol,
                        'p50_lta': max(2.0, p50_val),
                        'p90_lta': max(p50_val + 2.0, p90_val),
                        'is_lta_alert': is_lta_alert,
                        'cancel_user_pct': u_cancel,
                        'cancel_driver_pct': d_cancel,
                        'cancel_system_pct': s_cancel,
                        'cancel_total_pct': tot_cancel,
                        'is_cancel_alert': is_cancel_alert,
                        'breach_volume': breach_vol,
                        'breach_pct': breach_pct,
                        'avg_pickup_dist': pickup_dist,
                        'driver_reject_rate': reject_rate
                    })
                    
    return pd.DataFrame(records)

BASE_DF = generate_base_data_2025()

# ==========================================
# 3. RENDERERS
# ==========================================

# --- TAB 1: FIRST MILE LTA DASHBOARD ---
def render_chart_1_lta(df):
    agg_df = df.groupby(['date', 'service_id']).agg(
        p50_lta=('p50_lta', 'median'),
        p90_lta=('p90_lta', 'max'),
        is_lta_alert=('is_lta_alert', 'max')
    ).reset_index()
    
    total_records = len(agg_df)
    alert_count = int(agg_df['is_lta_alert'].sum())
    sla_compliance = round(100 - (alert_count / total_records * 100), 1) if total_records > 0 else 100.0
    breach_rate = round(100.0 - sla_compliance, 1)

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-green">
            <div class="kpi-title">SLA COMPLIANCE (ACCEPT ➔ BOARD)</div>
            <div class="kpi-value-green">{sla_compliance}% 🟢</div>
            <div class="kpi-sub">Chuẩn vận hành chặng đầu TP.HCM</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">CẢNH BÁO RED ALERT (P90 SLOW)</div>
            <div class="kpi-value-red">{alert_count:,} Sự cố 🔴</div>
            <div class="kpi-sub">Vượt Tipping Point LTA Boarding</div>
        </div>
        <div class="kpi-card kpi-card-purple">
            <div class="kpi-title">TỶ LỆ VỠ LTA CHẶNG ĐẦU</div>
            <div class="kpi-value" style="color:#8E44AD;">{breach_rate}% 🟣</div>
            <div class="kpi-sub">Đơn trễ First-Mile Pickup</div>
        </div>
    </div>
    """

    fig = make_subplots(
        rows=2, cols=2, 
        subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in SERVICE_CONFIG.values()], 
        vertical_spacing=0.15, horizontal_spacing=0.08
    )

    for s_id, cfg in SERVICE_CONFIG.items():
        row, col = cfg['grid_pos']
        sub_df = agg_df[agg_df['service_id'] == s_id].sort_values('date')
        
        fig.add_trace(go.Scatter(
            x=sub_df['date'], y=sub_df['p50_lta'], name='P50 (Median)', 
            mode='lines', line=dict(color=COLOR_SUCCESS, width=2), 
            showlegend=(row==1 and col==1)
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=sub_df['date'], y=sub_df['p90_lta'], name='P90 (Tail-Risk)', 
            mode='lines', line=dict(color=COLOR_PRIMARY, width=2), 
            showlegend=(row==1 and col==1)
        ), row=row, col=col)
        
        alert_df = sub_df[sub_df['is_lta_alert'] == True]
        if not alert_df.empty:
            fig.add_trace(go.Scatter(
                x=alert_df['date'], y=alert_df['p90_lta'], name='Sự cố Red Alert',
                mode='markers', marker=dict(symbol='x-open', size=8, color='#800000', line=dict(width=2)),
                showlegend=(row==1 and col==1)
            ), row=row, col=col)

        fig.add_hline(
            y=cfg['tipping_point'], line_dash="dash", line_color=COLOR_DANGER, 
            annotation_text=f"Alert (> {cfg['tipping_point']}m)", annotation_position="top right",
            annotation_font=dict(color=COLOR_DANGER, size=9), row=row, col=col
        )

    fig.update_layout(
        height=500, template="plotly_white", font=dict(family=FONT_FAMILY), 
        margin=dict(t=30, b=20, l=10, r=10), 
        legend=dict(orientation="h", y=1.08, x=0.6)
    )
    fig.update_xaxes(tickformat="%b %Y")
    fig.update_yaxes(title_text="Phút")
    return kpi_html, fig


# --- TAB 2: PRE-BOARDING CANCEL BREAKDOWN ---
def render_chart_2_cancel(df):
    agg_df = df.groupby(['date', 'service_id']).agg(
        cancel_user_pct=('cancel_user_pct', 'mean'),
        cancel_driver_pct=('cancel_driver_pct', 'mean'),
        cancel_system_pct=('cancel_system_pct', 'mean'),
        cancel_total_pct=('cancel_total_pct', 'mean'),
        is_cancel_alert=('is_cancel_alert', 'max')
    ).reset_index()

    avg_user = round(agg_df['cancel_user_pct'].mean(), 2)
    avg_driver = round(agg_df['cancel_driver_pct'].mean(), 2)
    avg_sys = round(agg_df['cancel_system_pct'].mean(), 2)
    avg_tot = round(avg_user + avg_driver + avg_sys, 2)
    alert_days = int(agg_df[agg_df['is_cancel_alert'] == True]['date'].nunique())

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-black">
            <div class="kpi-title">TỔNG HỦY PRE-BOARDING</div>
            <div class="kpi-value">{avg_tot}%</div>
            <div class="kpi-sub">Giai đoạn Accept ➔ Boarding</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">USER CANCEL (CHỜ LÂU)</div>
            <div class="kpi-value-red">{avg_user}%</div>
            <div class="kpi-sub">Khách hủy do xế tới chậm</div>
        </div>
        <div class="kpi-card kpi-card-orange">
            <div class="kpi-title">DRIVER CANCEL (TỪ CHỐI)</div>
            <div class="kpi-value" style="color:#F39C12;">{avg_driver}%</div>
            <div class="kpi-sub">Tài xế hủy sau khi nhận</div>
        </div>
        <div class="kpi-card kpi-card-purple">
            <div class="kpi-title">SYSTEM CANCEL (TIMEOUT)</div>
            <div class="kpi-value" style="color:#8E44AD;">{avg_sys}%</div>
            <div class="kpi-sub">Hệ thống hủy do quá giờ</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">NGÀY CẢNH BÁO CỦA NĂM</div>
            <div class="kpi-value-red">{alert_days} Ngày 🚨</div>
            <div class="kpi-sub">Vượt hạn mức hủy cho phép</div>
        </div>
    </div>
    """

    fig = make_subplots(
        rows=2, cols=2, 
        subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in SERVICE_CONFIG.values()], 
        vertical_spacing=0.15, horizontal_spacing=0.08
    )

    for s_id, cfg in SERVICE_CONFIG.items():
        row, col = cfg['grid_pos']
        sub_df = agg_df[agg_df['service_id'] == s_id].sort_values('date')
        
        fig.add_trace(go.Bar(x=sub_df['date'], y=sub_df['cancel_user_pct'], name='User Cancel (%)', marker_color=COLOR_DANGER, showlegend=(row==1 and col==1)), row=row, col=col)
        fig.add_trace(go.Bar(x=sub_df['date'], y=sub_df['cancel_driver_pct'], name='Driver Cancel (%)', marker_color=COLOR_WARNING, showlegend=(row==1 and col==1)), row=row, col=col)
        fig.add_trace(go.Bar(x=sub_df['date'], y=sub_df['cancel_system_pct'], name='System Cancel (%)', marker_color=COLOR_PURPLE, showlegend=(row==1 and col==1)), row=row, col=col)
        fig.add_trace(go.Scatter(x=sub_df['date'], y=sub_df['cancel_total_pct'], name='Tổng % Hủy', mode='lines', line=dict(color=COLOR_NAVY, width=1.5), showlegend=(row==1 and col==1)), row=row, col=col)

    fig.update_layout(
        barmode='stack', height=500, template="plotly_white", font=dict(family=FONT_FAMILY), 
        margin=dict(t=30, b=20, l=10, r=10), 
        legend=dict(orientation="h", y=1.08, x=0.3)
    )
    fig.update_xaxes(tickformat="%b %Y")
    fig.update_yaxes(title_text="% Hủy")

    acc = Accordion(children=[
        HTML("<p style='padding:8px;'>Phân tích Survival Analysis cho thấy khách hàng TP.HCM bắt đầu thoát app nếu tài xế không di chuyển tới điểm lấy hàng trong vòng <b>4.5 phút</b> sau khi accept.</p>"),
        HTML("<p style='padding:8px;'>Quy trình Ops 2025: Tự động kích hoạt Surge Fee 3k - 5k/đơn tại vùng đỏ để thu hút tài xế di chuyển nhanh tới điểm lấy hàng.</p>")
    ])
    acc.set_title(0, "▶ 💡 Survival Analysis & Điểm Kiên Nhẫn Khách Hàng (4.5 Phút)")
    acc.set_title(1, "▶ 💡 Ops Action Plan Khắc Phục Tỷ Lệ Hủy Chặng Đầu")

    return kpi_html, fig, acc


# --- TAB 3: VOLUME & HOTSPOT MATRIX ---
def render_chart_3_hotspot(df, scenario='Tất Cả'):
    agg_df = df.groupby(['district', 'service_id']).agg(
        total_vol=('volume', 'sum'),
        breach_vol=('breach_volume', 'sum')
    ).reset_index()
    
    if scenario == 'Mưa Lớn / Ngập Lụt':
        agg_df['breach_vol'] = (agg_df['breach_vol'] * 1.4).astype(int)
    elif scenario == 'Giờ Cao Điểm Sáng':
        agg_df['breach_vol'] = (agg_df['breach_vol'] * 1.25).astype(int)

    agg_df['breach_pct'] = np.clip((agg_df['breach_vol'] / agg_df['total_vol'] * 100).round(1), 0, 100)
    
    dist_totals = agg_df.groupby('district')['total_vol'].sum().sort_values(ascending=True)
    sorted_districts = dist_totals.index
    
    tot_vol_all = int(agg_df['total_vol'].sum())
    tot_breach_all = int(agg_df['breach_vol'].sum())
    overall_breach_pct = round((tot_breach_all / tot_vol_all * 100), 1) if tot_vol_all > 0 else 0

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-black">
            <div class="kpi-title">KHỐI LƯỢNG ĐƠN TỔNG (VOLUME)</div>
            <div class="kpi-value">{tot_vol_all:,} đơn</div>
            <div class="kpi-sub">Phạm vi bộ lọc đang chọn</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">ĐƠN BỊ TRỄ LTA BOARDING</div>
            <div class="kpi-value-red">{tot_breach_all:,} đơn</div>
            <div class="kpi-sub">Tỷ lệ vỡ SLA chặng đầu: <b>{overall_breach_pct}%</b></div>
        </div>
        <div class="kpi-card kpi-card-orange">
            <div class="kpi-title">KHU VỰC ĐỈNH ĐIỂM MA SÁT</div>
            <div class="kpi-value" style="font-size:16px; color:#C0392B;">🚨 Tân Bình, Q1 & Q10</div>
            <div class="kpi-sub">Tập trung đơn Siêu Tốc trễ điểm lấy hàng do kẹt xe & gửi xe lâu</div>
        </div>
    </div>
    """

    matrix = pd.pivot_table(agg_df, values='breach_pct', index='district', columns='service_id').reindex(sorted_districts).fillna(0)
    
    y_labels = [f"{d} ({dist_totals[d]/1000:.1f}K)" for d in sorted_districts]
    x_labels = [SERVICE_CONFIG[s]['name'] for s in ['SIEU_TOC', 'NHANH', '4H', 'DONG_GIA']]
    
    z_data = matrix[['SIEU_TOC', 'NHANH', '4H', 'DONG_GIA']].values

    fig = go.Figure(data=go.Heatmap(
        z=z_data, x=x_labels, y=y_labels, 
        colorscale='OrRd', texttemplate="%{z:.1f}%", textfont={"color": "white", "size": 10},
        colorbar=dict(title="SLA Breach %")
    ))

    fig.update_layout(
        title="<b>HOTSPOT MATRIX: TỶ LỆ TRỄ SLA BOARDING THEO QUẬN/HUYỆN & DỊCH VỤ (2025)</b>",
        height=620, template="plotly_white", font=dict(family=FONT_FAMILY), 
        margin=dict(t=50, b=20, l=160, r=20)
    )

    acc = Accordion(children=[
        HTML("<p style='padding:8px;'>Ma trận giúp phân biệt Hotspot do Volume quá tải (Tân Bình, Q1) hay do Lỗi Vận Hành/Ngâm đơn (Củ Chi, Q7).</p>")
    ])
    acc.set_title(0, "▶ 💡 Chuẩn Hóa Metrics (Normalize Metrics) Theo Khối Lượng Đơn")

    return kpi_html, fig, acc


# --- TAB 4: DISPATCHING DYNAMICS (EXACT MATCH WITH SCREENSHOTS 1 & 2) ---
def render_chart_4_dispatch(df, service_group_filter, scenario):
    # 24 Hours baseline arrays matching exact KPI values (Avg Pickup 1.34km, Max Reject 26% at 17:00)
    hours_24 = [f"{h:02d}:00" for h in range(24)]
    
    base_pickup = [1.0, 0.9, 0.8, 0.8, 0.9, 1.2, 1.5, 1.6, 1.8, 1.5, 1.4, 1.5, 1.5, 1.4, 1.3, 1.4, 1.6, 1.9, 1.6, 1.5, 1.3, 1.2, 1.1, 1.0]
    base_reject = [11, 9, 7, 7, 8, 13, 17, 21, 23, 19, 18, 20, 22, 18, 17, 19, 23, 26, 21, 17, 15, 13, 12, 10]

    # Adjust slightly for Stress Test scenario
    if scenario == 'Mưa/Giờ Cao Điểm (Stress Test)':
        dist_24 = [round(x * 1.3, 2) for x in base_pickup]
        reject_24 = [min(80, int(x * 1.35)) for x in base_reject]
    else:
        dist_24 = base_pickup
        reject_24 = base_reject

    avg_pickup = round(np.mean(dist_24), 2)
    max_pickup = round(np.max(dist_24), 1)
    max_reject = int(np.max(reject_24))
    peak_hour_idx = np.argmax(reject_24)
    peak_hour = f"{peak_hour_idx:02d}:00"

    # KPI Layout exactly matching Screenshots 1 & 2
    if scenario == 'Mưa/Giờ Cao Điểm (Stress Test)':
        diag_title = "🚨 Mất Cân Bằng Cung Cầu"
        diag_color = COLOR_DANGER
        ops_action = "Mở rộng bán kính gán đơn (Radius > 2.5km) & Tăng Surge Fee."
    else:
        diag_title = "✅ Vận Hành Ổn Định"
        diag_color = COLOR_SUCCESS
        ops_action = "Giữ nguyên tham số gán đơn (Radius 1.5km)."

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-green">
            <div class="kpi-title">AVG PICKUP DISTANCE</div>
            <div class="kpi-value" style="font-size: 24px; font-weight: 800;">{avg_pickup} km</div>
            <div class="kpi-sub">Đỉnh điểm: {max_pickup}km</div>
        </div>
        <div class="kpi-card kpi-card-orange">
            <div class="kpi-title">MAX REJECTION RATE</div>
            <div class="kpi-value" style="color: #F39C12; font-size: 24px; font-weight: 800;">{max_reject}%</div>
            <div class="kpi-sub">Lúc: {peak_hour}</div>
        </div>
        <div class="kpi-card kpi-card-green" style="border-left: 5px solid {diag_color};">
            <div class="kpi-title">CHẨN ĐOÁN (ROOT CAUSE)</div>
            <div class="kpi-value" style="color: {diag_color}; font-size: 18px; font-weight: 700;">{diag_title}</div>
            <div class="kpi-sub">➔ Ops Action: {ops_action}</div>
        </div>
    </div>
    """

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Blue Bar Chart for Pickup Distance
    fig.add_trace(
        go.Bar(
            x=hours_24, y=dist_24, 
            name="Avg Pickup Distance (km)", 
            marker_color=COLOR_BLUE_BAR, opacity=0.85
        ), secondary_y=False
    )
    
    # Orange Line Chart with Markers for Driver Reject Rate
    fig.add_trace(
        go.Scatter(
            x=hours_24, y=reject_24, 
            name="Driver Reject Rate (%)", 
            mode='lines+markers', 
            line=dict(color="#ED7D31", width=2.5),
            marker=dict(size=6, color="#ED7D31")
        ), secondary_y=True
    )
    
    # Target Lines with exact labels from Screenshots
    fig.add_hline(
        y=1.5, line_dash="dash", line_color=COLOR_SUCCESS, 
        annotation_text="Target Pickup (1.5km)", annotation_position="bottom right",
        annotation_font=dict(color=COLOR_SUCCESS, size=10), secondary_y=False
    )
    fig.add_hline(
        y=25, line_dash="dash", line_color=COLOR_DANGER, 
        annotation_text="Warning Reject (25%)", annotation_position="top right",
        annotation_font=dict(color=COLOR_DANGER, size=10), secondary_y=True
    )

    fig.update_layout(
        title=dict(text="<b>PHÂN TÍCH LOGIC ĐIỀU PHỐI (DISPATCHING DYNAMICS)</b>", font=dict(size=13, color="#1A202C")),
        height=460, template="plotly_white", font=dict(family=FONT_FAMILY), 
        margin=dict(t=40, b=30, l=40, r=40),
        xaxis=dict(title="<b>Khung Giờ Trong Ngày</b>", showgrid=True, gridcolor="#F0F0F0"),
        yaxis=dict(title="<b>Khoảng cách Pickup (km)</b>", range=[0, 25], showgrid=True, gridcolor="#F0F0F0"),
        yaxis2=dict(title="<b>Tỷ lệ Từ chối (%)</b>", range=[10, 27], showgrid=False),
        legend=dict(orientation="h", y=1.08, x=0.65, font=dict(size=11))
    )
    return kpi_html, fig


# --- TAB 5: SPATIAL ANOMALY SCATTER ---
def render_chart_5_scatter(df, view_mode):
    total_volume = int(df['volume'].sum())
    sample_size = min(1500, max(200, int(total_volume / 100)))
    
    np.random.seed(42)
    n_od = int(sample_size * 0.5)
    dist_od = np.random.uniform(0.2, 4.5, n_od)
    ttp_od = 2.5 * dist_od + np.random.normal(4, 1.2, n_od)
    fraud_idx = np.random.choice(n_od, max(1, int(n_od * 0.18)), replace=False)
    dist_od[fraud_idx] = np.random.uniform(0.2, 1.2, len(fraud_idx))
    ttp_od[fraud_idx] = np.random.uniform(16, 48, len(fraud_idx))
    
    n_bt = sample_size - n_od
    dist_bt = np.random.uniform(0.2, 4.5, n_bt)
    ttp_bt = 22 + 2.5 * dist_bt + np.random.normal(4, 1.8, n_bt)
    sloth_idx = np.random.choice(n_bt, max(1, int(n_bt * 0.20)), replace=False)
    dist_bt[sloth_idx] = np.random.uniform(0.2, 1.8, len(sloth_idx))
    ttp_bt[sloth_idx] = np.random.uniform(45, 95, len(sloth_idx))

    is_fraud_od = np.zeros(n_od, dtype=bool); is_fraud_od[fraud_idx] = True
    is_sloth_bt = np.zeros(n_bt, dtype=bool); is_sloth_bt[sloth_idx] = True

    total_red = len(fraud_idx) + len(sloth_idx)
    pct_red = round((total_red / sample_size) * 100, 1)

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-black">
            <div class="kpi-title">MẪU ĐƠN PHÂN TÍCH SPATIAL</div>
            <div class="kpi-value">{sample_size:,} mẫu đơn</div>
            <div class="kpi-sub">Trích xuất từ <b>{total_volume:,}</b> đơn trong bộ lọc</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">PHÁT HIỆN RED ZONE (MULTI-APP/NGÂM ĐƠN)</div>
            <div class="kpi-value-red">{total_red:,} đơn ({pct_red}%)</div>
            <div class="kpi-sub">Đơn gần (<1.5km) nhưng tới điểm lấy quá trễ</div>
        </div>
        <div class="kpi-card kpi-card-orange">
            <div class="kpi-title">OPS DIAGNOSIS CHẶNG BOARDING</div>
            <div class="kpi-value" style="font-size:15px; color:#C0392B;">🚨 Báo động hành vi chạy chui app ngoài</div>
            <div class="kpi-sub">Tài xế bấm nhận đơn AhaMove nhưng di chuyển làm đơn nền tảng khác trước khi tới Boarding.</div>
        </div>
    </div>
    """

    fig = make_subplots(rows=1, cols=2, subplot_titles=("<b>PANEL A: ON-DEMAND (Siêu Tốc / Nhanh)</b>", "<b>PANEL B: BATCHING (4H / Đồng Giá)</b>"))

    show_normal = view_mode in ['Tất Cả Đơn Hàng', 'Chỉ Đơn Bình Thường']
    show_violation = view_mode in ['Tất Cả Đơn Hàng', 'Chỉ Đơn Vi Phạm Red Zone']

    if show_normal:
        fig.add_trace(go.Scatter(x=dist_od[~is_fraud_od], y=ttp_od[~is_fraud_od], mode='markers', marker=dict(color='#3498DB', opacity=0.5, size=5), name="Đơn Bình Thường"), row=1, col=1)
        fig.add_trace(go.Scatter(x=dist_bt[~is_sloth_bt], y=ttp_bt[~is_sloth_bt], mode='markers', marker=dict(color='#3498DB', opacity=0.5, size=5), name="Đơn Bình Thường", showlegend=False), row=1, col=2)

    if show_violation:
        fig.add_trace(go.Scatter(x=dist_od[is_fraud_od], y=ttp_od[is_fraud_od], mode='markers', marker=dict(color=COLOR_DANGER, opacity=0.8, size=6), name="Gian Lận Red Zone"), row=1, col=1)
        fig.add_trace(go.Scatter(x=dist_bt[is_sloth_bt], y=ttp_bt[is_sloth_bt], mode='markers', marker=dict(color=COLOR_DANGER, opacity=0.8, size=6), name="Gian Lận Red Zone", showlegend=False), row=1, col=2)

    fig.add_shape(type="rect", x0=0, y0=15, x1=1.5, y1=100, fillcolor="red", opacity=0.1, line=dict(color="red", dash="dash"), row=1, col=1)
    fig.add_shape(type="rect", x0=0, y0=45, x1=2.0, y1=100, fillcolor="red", opacity=0.1, line=dict(color="red", dash="dash"), row=1, col=2)

    fig.update_layout(height=460, template="plotly_white", font=dict(family=FONT_FAMILY), margin=dict(t=40, b=30, l=30, r=10))
    fig.update_xaxes(title_text="Khoảng cách Accept -> Board (Km)")
    fig.update_yaxes(title_text="Thời gian Boarding - TTP (Phút)")
    return kpi_html, fig


# --- TAB 6: B2B MERCHANT FRICTION (EXACT MATCH WITH SCREENSHOT 3) ---
def render_chart_6_b2b(df, b2b_group_filter, current_district):
    is_batching = '4H' in b2b_group_filter or 'Batching' in b2b_group_filter
    sla_target = 45.0 if is_batching else 15.0
    time_key = 'time_batching' if is_batching else 'time_ondemand'
    
    # Filter Merchants based on Global District Filter
    if current_district != 'ALL':
        merchants = [m for m in B2B_MERCHANTS_MASTER if m['district'] == current_district]
        if not merchants:
            merchants = B2B_MERCHANTS_MASTER
    else:
        merchants = B2B_MERCHANTS_MASTER

    m_data = []
    for m in merchants:
        m_data.append({"name": m['name'], "time": m[time_key], "district": m['district']})

    df_b2b = pd.DataFrame(m_data).sort_values('time', ascending=True)
    
    total_hubs = len(df_b2b)
    over_sla_cnt = sum(1 for t in df_b2b['time'] if t > sla_target)
    pct_over = int((over_sla_cnt / total_hubs) * 100) if total_hubs > 0 else 0
    avg_group_time = round(df_b2b['time'].mean(), 1) if total_hubs > 0 else 0
    worst_m = df_b2b.iloc[-1]
    worst_diff = round(worst_m['time'] - sla_target, 1)
    diff_sign = f"+{worst_diff}" if worst_diff > 0 else f"{worst_diff}"

    kpi_html = f"""
    <div class="card-container">
        <div class="kpi-card kpi-card-black">
            <div class="kpi-title">SỐ LƯỢNG PARTNER B2B</div>
            <div class="kpi-value" style="font-size: 24px; font-weight: 800;">{total_hubs} Kho/Điểm</div>
            <div class="kpi-sub">📍 Dispatch Distance < 2km</div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">TỶ LỆ VƯỢT CHUẨN SLA ({int(sla_target)} PHÚT)</div>
            <div class="kpi-value-red" style="font-size: 24px; font-weight: 800;">{pct_over}% <span style="font-size: 13px; color: #718096; font-weight: normal;">({over_sla_cnt}/{total_hubs} kho)</span></div>
            <div class="kpi-sub">TTB TB Nhóm: <b>{avg_group_time} phút</b></div>
        </div>
        <div class="kpi-card kpi-card-red">
            <div class="kpi-title">ĐIỂM NGHỄN B2B BÁO ĐỘNG NHẤT</div>
            <div class="kpi-value" style="font-size: 15px; color: #E74C3C; font-weight: 800; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;">🚨 {worst_m['name']}</div>
            <div class="kpi-sub">Thời gian Board: <b>{worst_m['time']} phút</b> (Vượt {diff_sign}p so với chuẩn).</div>
        </div>
    </div>
    """

    colors = [COLOR_DANGER if t > sla_target else "#7F8C8D" for t in df_b2b['time']]
    
    fig = go.Figure(go.Bar(
        x=df_b2b['time'], y=df_b2b['name'], orientation='h',
        marker_color=colors, 
        text=[f"{t} phút" for t in df_b2b['time']], 
        textposition="outside",
        textfont=dict(size=11)
    ))
    
    target_label = f"Chuẩn 4H ({int(sla_target)} phút)" if is_batching else f"Chuẩn SLA ({int(sla_target)} phút)"
    
    fig.add_vline(
        x=sla_target, line_dash="dash", line_color=COLOR_DANGER, line_width=1.5,
        annotation_text=target_label, annotation_position="top",
        annotation_font=dict(color=COLOR_DANGER, size=11, family=FONT_FAMILY)
    )

    fig.update_layout(
        title=dict(
            text="<b>CHART 3.2: B2B MERCHANT BOARDING FRICTION (XẾP HẠNG MA SÁT ĐIỂM LẤY HÀNG)</b><br><span style='font-size:11px; color:#718096; font-weight:normal;'>Lọc các đơn có Dispatch Distance < 2km</span>",
            font=dict(size=13, color="#1A202C")
        ),
        height=480, template="plotly_white", font=dict(family=FONT_FAMILY), 
        margin=dict(t=50, b=30, l=230, r=50),
        xaxis=dict(title="<b>Thời Gian Trung Bình Từ Accept Đến Board - Avg Time to Board (Phút)</b>", range=[0, max(55, max(df_b2b['time']) + 8)], showgrid=True, gridcolor="#F0F0F0"),
        yaxis=dict(title="<b>Đối Tác / Kho Hàng B2B</b>", showgrid=False)
    )
    return kpi_html, fig


# ==========================================
# 4. CONTROLLERS & DASHBOARD INTERACTIVITY
# ==========================================

date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2025, 1, 1), layout=Layout(width='20%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2025, 12, 31), layout=Layout(width='20%'))
district_widget = widgets.Dropdown(options=['ALL'] + HCM_DISTRICTS, value='ALL', description='Khu vực:', layout=Layout(width='20%'))
hour_slot_widget = widgets.Dropdown(options=HOUR_SLOTS, value='ALL', description='Khung giờ:', layout=Layout(width='20%'))
btn_refresh = widgets.Button(description='🔁 Cập Nhật Dashboard', layout=Layout(width='18%'))
btn_refresh.add_class('action-btn-orange')

# Sub-Filters matching exact Option Text in Screenshots
dd_c3_ops = widgets.Dropdown(options=['Tất Cả', 'Mưa Lớn / Ngập Lụt', 'Giờ Cao Điểm Sáng'], value='Tất Cả', description='Kịch Bản Ops:', layout=Layout(width='35%'))

dd_c4_service = widgets.Dropdown(
    options=['Nhóm 1: Tức Thời (On-Demand) - Siêu Tốc/Nhanh', 'Nhóm 2: Gom Chuyến (Batching) - 4H/Đồng Giá'], 
    value='Nhóm 2: Gom Chuyến (Batching) - 4H/Đồng Giá', 
    description='Dịch Vụ:', layout=Layout(width='45%')
)
dd_c4_scenario = widgets.Dropdown(
    options=['Bình Thường (Baseline)', 'Mưa/Giờ Cao Điểm (Stress Test)'], 
    value='Bình Thường (Baseline)', 
    description='Tình huống:', layout=Layout(width='45%')
)

dd_c5_view = widgets.Dropdown(options=['Tất Cả Đơn Hàng', 'Chỉ Đơn Vi Phạm Red Zone', 'Chỉ Đơn Bình Thường'], value='Tất Cả Đơn Hàng', description='Chế Độ Xem:', layout=Layout(width='40%'))

dd_c6_group = widgets.Dropdown(
    options=['4H / Đồng Giá (Batching)', 'Siêu Tốc / Nhanh (On-Demand)'], 
    value='4H / Đồng Giá (Batching)', 
    description='Nhóm Dịch Vụ:', layout=Layout(width='38%')
)

# Output Containers
out_lta = Output()
out_cancel = Output()
out_hotspot = Output()
out_dispatch = Output()
out_scatter = Output()
out_b2b = Output()

def get_filtered_df():
    start = pd.to_datetime(date_start_widget.value)
    end = pd.to_datetime(date_end_widget.value)
    dist = district_widget.value
    hr = hour_slot_widget.value
    
    mask = (BASE_DF['date'] >= start) & (BASE_DF['date'] <= end)
    if dist != 'ALL': mask &= (BASE_DF['district'] == dist)
    if hr != 'ALL': mask &= (BASE_DF['hour_slot'] == hr)
    
    res = BASE_DF[mask]
    return res if not res.empty else BASE_DF

def update_global_dashboard(b=None):
    filtered_df = get_filtered_df()

    with out_lta:
        out_lta.clear_output(wait=True)
        kpi_h, fig = render_chart_1_lta(filtered_df)
        display(HTML(kpi_h)); display(fig)

    with out_cancel:
        out_cancel.clear_output(wait=True)
        kpi_h, fig, acc = render_chart_2_cancel(filtered_df)
        display(HTML(kpi_h)); display(fig); display(acc)

    update_c3(filtered_df=filtered_df)
    update_c4(filtered_df=filtered_df)
    update_c5(filtered_df=filtered_df)
    update_c6(filtered_df=filtered_df)

def update_c3(change=None, filtered_df=None):
    if filtered_df is None: filtered_df = get_filtered_df()
    with out_hotspot:
        out_hotspot.clear_output(wait=True)
        kpi_h, fig, acc = render_chart_3_hotspot(filtered_df, dd_c3_ops.value)
        display(HTML(kpi_h)); display(fig); display(acc)

def update_c4(change=None, filtered_df=None):
    if filtered_df is None: filtered_df = get_filtered_df()
    with out_dispatch:
        out_dispatch.clear_output(wait=True)
        kpi_h, fig = render_chart_4_dispatch(filtered_df, dd_c4_service.value, dd_c4_scenario.value)
        display(HTML(kpi_h)); display(fig)

def update_c5(change=None, filtered_df=None):
    if filtered_df is None: filtered_df = get_filtered_df()
    with out_scatter:
        out_scatter.clear_output(wait=True)
        kpi_h, fig = render_chart_5_scatter(filtered_df, dd_c5_view.value)
        display(HTML(kpi_h)); display(fig)

def update_c6(change=None, filtered_df=None):
    if filtered_df is None: filtered_df = get_filtered_df()
    with out_b2b:
        out_b2b.clear_output(wait=True)
        kpi_h, fig = render_chart_6_b2b(filtered_df, dd_c6_group.value, district_widget.value)
        display(HTML(kpi_h)); display(fig)

# Event Handlers
btn_refresh.on_click(update_global_dashboard)
dd_c3_ops.observe(lambda c: update_c3(), names='value')
dd_c4_service.observe(lambda c: update_c4(), names='value')
dd_c4_scenario.observe(lambda c: update_c4(), names='value')
dd_c5_view.observe(lambda c: update_c5(), names='value')
dd_c6_group.observe(lambda c: update_c6(), names='value')

# Layout Assembly
header = HTML(f"""
<div style="background-color: {COLOR_NAVY}; padding: 14px 18px; border-radius: 8px; margin-bottom: 12px; font-family: {FONT_FAMILY}; display: flex; justify-content: space-between; align-items: center;">
    <div>
        <h2 style="color: {COLOR_PRIMARY}; margin: 0; font-size: 20px;">🚀 AhaMove SGN - Cấp độ Phân Tích Logic Thuật Toán Điều Phối</h2>
        <p style="color: #A0AEC0; margin: 4px 0 0 0; font-size: 12px;">Chuyên sâu Phase Accept ➔ Boarding (Lấy Hàng) | TP. Hồ Chí Minh (1/1/2025 - 31/12/2025)</p>
    </div>
    <span style="background: rgba(255,107,0,0.2); color: #FF6B00; padding: 4px 10px; border-radius: 4px; font-size: 11px; font-weight: 700;">SENIOR DA EDITION v2025</span>
</div>
""")

banner_b2b_global = HTML('<div class="global-filter-banner">🔒 <b>Global Filter Active:</b> Dispatch Distance < 2.0 km (Circuity Factor 1.3)</div>')

tab = Tab(children=[
    VBox([out_lta]),
    VBox([out_cancel]),
    VBox([HBox([dd_c3_ops]), out_hotspot]),
    VBox([HBox([dd_c4_service, dd_c4_scenario]), out_dispatch]),
    VBox([HBox([dd_c5_view]), out_scatter]),
    VBox([HBox([dd_c6_group, banner_b2b_global]), out_b2b])
])

tab.set_title(0, '1. LTA Dashboard (SLA)')
tab.set_title(1, '2. Cancel Rate Breakdown')
tab.set_title(2, '3. Hotspot Matrix')
tab.set_title(3, '4. Dispatch Dynamics')
tab.set_title(4, '5. Spatial Anomaly Scatter')
tab.set_title(5, '6. B2B Merchant Friction')

# Display Dashboard UI
display(CSS_STYLING)
display(header)
display(HBox([date_start_widget, date_end_widget, district_widget, hour_slot_widget, btn_refresh]))
display(tab)

# Trigger Initial Render
update_global_dashboard()

HTML(value="\n<style>\n    .card-container { display: flex; gap: 12px; margin-bottom: 12px; font-family: -appl…

HTML(value='\n<div style="background-color: #0B192C; padding: 14px 18px; border-radius: 8px; margin-bottom: 12…